<a id="inicio"></a>
<br><br><br>
<div align="left">
<h1><font size=6>Pandas</font></h1>
    <h2><font color="#5E5E5E" size=5>A1. MISC  </h2>
<br><br><br><br>
</div>


<div align="right">
<font color="#B51700" size=4><b>Análisis y visualización de datos</b></font><br><br>
<font  size=3>Máster Universitario en Ingeniería Informática</font><br>
<font  size=3>Máster Universitario en Big Data y Computación en la Nube</font><br>
<font size=3>Máster Universitario en Inteligencia Artificial</font><br><br>
<font color="#5E5E5E" size=3><b>Luis de la Ossa  &nbsp;&nbsp;</b></font><span style="background-color:#B51700; color:white; padding:2px 6px; font-family:Arial, sans-serif;font-weight:bold;font-size:18px;">UCLM</span>
</div>

---

In [ ]:
import pandas as pd
import numpy as np

pd.__version__

---

##  <font color="#5E5E5E"> Archivos "complicados" </font>

Muchas veces, los archivos `.csv` contienen elementos que dificultan la lectura directa, como por ejemplo comentarios, comillas, etc. La función `pd.read_csv()` incluye multitud de opciones que permiten tratar archivos que, a priori, no son **estrictamente** un `.csv`:

* `skiprows`. Número de líneas de la cabecera que han de ser ignoradas.
* `na_values`. Indica cómo se representan los valores perdidos.
* `names`. Nombres de las columnas.
* `usecols`. Indica qué columnas utilizar. 
* `skipfooter`. Número de líneas del final del archivo que deben ser descartadas.
* `decimal`. Separador decimal.
* `thousands`. Separador de miles.
* `comment`. Cómo se indica que una línea es un comentario.
* `true_values`. Cómo se representa el valor `True`
* `false_values`. Cómo se representa el valor `False`.

In [ ]:
#! cat data/awkward.csv

In [ ]:
columnas = ["Species", "Genome size", "GC%", "Genes", "Year", "Status", "Is animal", "Is plant"]

df = pd.read_csv( 
     "data/awkward.csv", 
    skiprows=11, 
    na_values=["-", "missing"], 
    names=columnas ,
    usecols=columnas ,
    skipfooter=1,
    decimal=",",
    thousands="_",
    comment="#",
    true_values=["True", "1", "yes", '"t"'],
    false_values=["False", "0", "no", '"f"'],
    dtype={"Is animal": bool, "Is plant": bool},
    engine='python') # El motor C no admite skipfooter.

df.info()

---

##  <font color="#5E5E5E"> Reducción del consumo en memoria </font>

In [ ]:
weather = pd.read_csv("data/all_weather.csv")
weather.info()

In [ ]:
weather.memory_usage()

`memory_usage(deep=True)` ofrece una estimación más completa de la memoria ocupada por cada columna y el índice.

In [ ]:
weather.memory_usage(deep=True) 

In [ ]:
weather.memory_usage(deep=True).sum()

Se puede reducir la memoria ocupada mediante algunas acciones:

* Incluir solamente las columnas necesarias
* Utilizar categorías
* Reducir la precisión numérica

In [ ]:
weather = pd.read_csv(
            "data/all_weather.csv",
            usecols=["City", "Year", "Day of year", "Rainfall (mm)", "Temperature (°C)"], 
            dtype={"City": "category", 
                   "Year": "int16", 
                   "Day of year": "int16",
                   "Rainfall (mm)": "Int16", 
                   "Temperature (°C)": "float32",
                  }
            )

print(weather.memory_usage(deep=True))

In [ ]:
weather.memory_usage(deep=True).sum()

In [ ]:
pd.read_csv("data/all_weather.csv").memory_usage(deep=True).sum()

---

##  <font color="#5E5E5E"> Lectura en partes </font>

El parámetro `chunksize` permite indicar que se lean los archivos en partes. Cada una se puede procesar por separado, aunque es necesario adaptar el código a esta circunstancia, y no siempre se puede.

In [ ]:
chunk_rainy_days_list = []

for chunk in pd.read_csv("data/all_weather.csv", chunksize=10000): 
    chunk_rainy_days = chunk[chunk["Rainfall (mm)"] > 100] 
    chunk_rainy_days_list.append(chunk_rainy_days)
    
all_chunk_rainy_days = pd.concat(chunk_rainy_days_list) 
all_chunk_rainy_days.head()

---


## <font color="#5E5E5E">Aumento de la eficiencia con *Pyarrow*</font>

pandas puede almacenar columnas mediante distintas representaciones, entre ellas NumPy y Apache Arrow. PyArrow proporciona acceso desde Python a las estructuras y operaciones de Arrow, orientadas al trabajo con datos por columnas.

En pandas 3.0, el texto inferido utiliza el dtype `str`, que se almacena mediante PyArrow cuando esta biblioteca está disponible. Además, `dtype_backend='pyarrow'` permite solicitar tipos respaldados por Arrow al leer los datos, incluidos los numéricos. Esto puede reducir el uso de memoria o acelerar determinadas operaciones, pero el resultado depende de los datos y del trabajo realizado.

Este artículo muestra una comparativa [(enlace)](https://towardsdatascience.com/utilizing-pyarrow-to-improve-pandas-and-dask-workflows-2891d3d96d2b).

In [ ]:
#%pip install pyarrow

In [ ]:
#df_titanic = pd.read_csv('data/titanic_prep.csv', index_col=0,  engine="pyarrow") # El motor que interpreta el CSV,  backend numpy
df_titanic = pd.read_csv('data/titanic_prep.csv', index_col=0,  dtype_backend="pyarrow") # Backend arrow
df_titanic.info()

---
## <font color="#5E5E5E">Encadenamiento de métodos</font>

En pandas, muchos métodos devuelven un nuevo DataFrame o una Serie modificada. Eso permite encadenar varias operaciones una tras otra.
    
El encadenamiento de métodos (*method chaining*) es una forma de escribir código más fluido, legible y conciso, aplicando múltiples operaciones sobre un DataFrame en una sola secuencia sin tener que crear variables intermedias.  

En general, el method chaining en pandas no mejora la eficiencia computacional (rendimiento), sino la eficiencia del desarrollo y mantenimiento de código.

In [ ]:
data = {
    'Nombre': [pd.NA, 'Mireia', 'Alberto', 'Pau', 'Sergio', pd.NA, 'Rafael', 'Fernando'],
    'Edad': [33.0, 27.0, 35.0, 37.0, 33.0, 33.0, 31.0, 33.0],
    'Deporte': [pd.NA, 'Natación', 'Ciclismo', pd.NA, 'Baloncesto', 'Motociclismo', 'Tenis', 'Fútbol']
}

index = ['Alonso', 'Belmonte', 'Contador', 'Gasol', 'Llul', 'Márquez', 'Nadal', 'Torres']

df = pd.DataFrame(data, index=index)

df

 Sin *method chaining*

In [ ]:
df_na = df.dropna()
df_30 = df_na.query("Edad > 30")
df_sorted = df_30.sort_values("Edad", ascending=False)
dfp = df_sorted.assign(Grupo=lambda x: np.where(x["Edad"] >= 35, "Senior", "Adulto"))
dfp

Con *method chaining*

In [ ]:
dfp = (
    df.dropna()
      .query("Edad > 30")
      .sort_values("Edad", ascending=False)
      .assign(Grupo=lambda x: np.where(x["Edad"] >= 35, "Senior", "Adulto"))
)
dfp

### <font color="#5E5E5E">`pipe`</font>

En muchos casos las funciones de procesamiento de *DataFrame* no se pueden encadenar de manera natural. El método `df.pipe(función, *args, **kwargs)` es equivalente a hacer `función(df, *args, **kwargs)`, y permite encadenar funciones, mejorando la legibilidad del código [**(+ inf)**](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pipe.html).

Este artículo presenta un ejemplo de aplicación de `Pipe` [(enlace)](https://sinyi-chou.github.io/python-pandas-pipe/). A continuación, se muestra el resumen.

In [ ]:
import numpy as np

df = pd.DataFrame({
    'name': ['Ted'] * 3 + ['Lisa'] * 3 + ['Sam'] * 3,
    'subject': ['math', 'physics', 'history'] * 3,
    'score_t': np.random.randint(60, 100, 9),
    'score_p': np.random.randint(60, 100, 9)
})

df

La siguiente función añade el ranking de cada alumno según su puntuación en el examen de teoría (`score_t`).

In [ ]:
def get_subject_rank(input_df):
    input_df = input_df.copy()
    input_df['rank_t'] = (input_df
                                .groupby(['subject'])['score_t']
                                .rank(ascending=False))
    return input_df

df.pipe(get_subject_rank)

Las funciones pueden aceptar otros parámetros. En este caso se calcula la nota final ponderando (dando un peso) a la teoría y la práctica.

In [ ]:
def final_score(input_df, weight_t, weight_p):
    input_df = input_df.assign(final_score=lambda df: weight_t*df['score_t']+weight_p*df['score_p'])
    return input_df

df.pipe(final_score, 0.6, 0.4)

Las dos funciones se pueden encadenar

In [ ]:
df_final = (
    df.pipe(get_subject_rank)
      .pipe(final_score, 0.6, 0.4)
)
df_final

Por defecto, `DataFrame.pipe()` "espera" que la función acepte el propio `DataFrame` como primer argumento. Si no es así, debe pasarse como tupla la función, y el nombre del parámetro receptor de `DataFrame`. En el ejemplo, `df_final.pipe((topk, 'data'), 5)` equivale a `topk(5, data=df_final)`.

In [ ]:
def topk(k, data):
    return data.nlargest(k, 'final_score')

#df_final.pipe(topk, 5)   # No funciona
df_final.pipe((topk,'data'), 5)

---

## <font color="#5E5E5E">Gráficas</font>

Pandas permite crear gráficas a partir de DataFrames y Series (usando Matplotlib internamente). La sintaxis general es:

```
df.plot(kind='tipo_de_gráfica', ...)
```

Donde `kind` puede ser:

* 'line' → línea (por defecto)
* 'bar' / 'barh' → barras / barras horizontales
* 'hist' → histograma
* 'box' → diagrama de caja
* 'scatter' → dispersión
* 'pie' → pastel


In [ ]:
df_titanic

In [ ]:
ax = df_titanic['Edad'].plot(kind='hist', bins=20, title='Distribución de edades', edgecolor='black');

ax.set_xlabel('Edad (años)')
ax.set_ylabel('Número de pasajeros');

In [ ]:
df_titanic.plot(kind='box', column='Edad', by='Clase');

---

## <font color="#5E5E5E">Estilos</font>

Cuando los *DataFrame* son un elemento que se ha de analizar, es de utilidad destacar valores mediante color. Pandas proporciona utilidades para modificar el aspecto con que se muestran los *DataFrame* [**(+ inf)**](https://pandas.pydata.org/pandas-docs/stable/user_guide/style.html).

In [ ]:
import seaborn as sns
cm = sns.light_palette("green", as_cmap=True)

(weather.head(10).style
    .background_gradient(
        cmap='Greens', subset=['Rainfall (mm)', 'Temperature (°C)']
    )
    .highlight_max(subset=['Rainfall (mm)', 'Temperature (°C)'])
    .set_caption('Primeras diez observaciones meteorológicas')
    .format({
        'Rainfall (mm)': '{:.0f}',
        'Temperature (°C)': '{:.1f}'
    }, na_rep='—'))

---


## <font color="#5E5E5E">El método `DataFrame.explode()`</font>

Cuando una columna contiene listas, crea una fila con cada elemento, duplicando el resto del `DataFrame`. 

In [ ]:
df = pd.DataFrame({'A': [[0, 1, 2], 'foo', [], [3, 4]],
                   'B': 1,
                   'C': [['a', 'b', 'c'], pd.NA, [], ['d','e']]})
df

In [ ]:
df.explode('A')

Se puede aplicar a varias columnas siempre y cuando los tamaños de las listas en cada una sean similares para todas las filas.

In [ ]:
df.explode(['A','C'])

---

## <font color="#5E5E5E">De JSON anidado a tabla</font>

`pd.json_normalize()` transforma estructuras anidadas en tablas. `record_path` indica dónde están los registros que deben convertirse en filas; `meta` permite conservar datos del objeto que contiene esos registros.




In [ ]:
pedidos_json = [
    {'pedido': 101, 
     'cliente': {'nombre': 'Ana'},
     'lineas': [{'producto': 'Libro', 'unidades': 2},
                {'producto': 'Cuaderno', 'unidades': 1}]},
    {'pedido': 102, 
     'cliente': {'nombre': 'Luis'},
     'lineas': [{'producto': 'Lápiz', 'unidades': 3}]}
]

lineas_pedido = pd.json_normalize(
    pedidos_json,
    record_path='lineas',
    meta=['pedido', ['cliente', 'nombre']]
)
lineas_pedido


---

## <font color="#5E5E5E">Opciones</font>

Pandas  permite configurar un número importante de opciones [**(+ inf)**](https://pandas.pydata.org/pandas-docs/stable/user_guide/options.html). Para ello, proporciona 4 funciones:

* `pd.get_option() / pd.set_option()`, que permite fijar el valor para una opción.
* `pd.reset_option()`, que resetea el valor de una opción al valor por defecto.
* `pd.describe_option()`, que devuelve la descripción de una opción.
* `pd.option_context()`, que permite ajustar las opciones para un bloque de código.

Algunas opciones de interés.

* `display.max_rows` y `display.max_columns`: número máximo de filas y columnas que se muestran.
* `display.max_colwidth`: máxima anchura de una columna.
* `display.precision`: precisión con que se muestran los decimales.

*  etc.


In [ ]:
pd.describe_option("display.expand_frame_repr")

In [ ]:
pd.set_option("display.precision", 3)

pd.set_option("display.expand_frame_repr", False)

pd.set_option("display.max_rows", 25)

---


## <font color="#5E5E5E">Testeo y comparación</font>

Pandas incorpora varias funciones para comprobar si varios objetos son totalmente iguales:
* `pandas.testing.assert_frame_equal()` que comprueba si dos `DataFrame` son iguales.
* `pandas.testing.assert_series_equal()` que comprueba si dos `Series` son iguales.
* `pandas.testing.assert_index_equal()` que comprueba si dos índices son iguales. 

Además de comprobar el contenido, pueden comprobar otros aspectos como tipos de datos, tipo de los índices, frecuencia en índices temporales, [( **+inf** )](https://pandas.pydata.org/pandas-docs/version/1.3/reference/api/pandas.testing.assert_frame_equal.html#).

In [ ]:
from pandas.testing import assert_frame_equal

df1 = pd.DataFrame({'a': [1, 2], 'b': [3, 4]})
df2 = pd.DataFrame({'a': [1, 2], 'b': [3.0, 4.0]})

#assert_frame_equal(df1,df2)   # No cumple porque el tipo de la columna b es distinto.
assert_frame_equal(df1,df2, check_dtype=False)

In [ ]:
df2 = pd.DataFrame({'a': [1, 2], 'c': [3.0, 4.0]})

# assert_frame_equal(df1,df2, check_dtype=False) No pasa porque los nombres de las columnas son diferentes.

In [ ]:
calculado = pd.DataFrame({'valor': [0.1 + 0.2]})
esperado = pd.DataFrame({'valor': [0.3]})

assert_frame_equal(calculado, esperado, check_exact=False, rtol=1e-12, atol=1e-12)


### <font color="#5E5E5E">Comparación de `DataFrames`</font>

In [ ]:
data = {
    'Nombre': [pd.NA, 'Mireia', 'Alberto', 'Pau', 'Sergio', pd.NA, 'Rafael', 'Fernando'],
    'Edad': [33.0, 27.0, 35.0, 37.0, 33.0, 33.0, 31.0, 33.0],
    'Deporte': [pd.NA, 'Natación', 'Ciclismo', pd.NA, 'Baloncesto', 'Motociclismo', 'Tenis', 'Fútbol']
}

index = ['Alonso', 'Belmonte', 'Contador', 'Gasol', 'Llul', 'Márquez', 'Nadal', 'Torres']

df = pd.DataFrame(data, index=index)

df

In [ ]:
df2 = df.copy()
df2.loc['Gasol','Deporte']='Baloncesto'
df2.loc['Nadal','Edad']=pd.NA
df2.loc['Alonso','Nombre']='Fernando'
df2

Para comparar los valores se utiliza `DataFrame.equals()`

In [ ]:
df.equals(df2)

    
### `DataFrame.compare()`

Este método permite comparar el contenido de dos `DataFrame` con las mismas etiquetas y en la misma posición en los índices de filas y columnas. Devuelve un nuevo `DataFrame` con la diferencia.

In [ ]:
df.compare(df2)

Por defecto, muestra las diferencias en las columnas. Las puede alinear en las filas con `align_axis=0` (el valor por defecto es uno).

In [ ]:
df.compare(df2, align_axis=0)

El parámetro `keep_equal=True` hace que en las filas y columnas implicadas (en las que hay diferencias) se muestren también los valores igual.

In [ ]:
df.compare(df2, keep_equal=True)

Con `keep_shape=True` se muestran todas las filas y columnas.

In [ ]:
df.compare(df2, keep_shape=True)
# df.compare(df2, keep_shape=True, keep_equal=True)

---


## <font color="#5E5E5E">Validación  previa al procesamiento</font>


Antes de procesar una tabla, pueden comprobarse las condiciones que necesita el análisis: columnas obligatorias, claves únicas, valores ausentes y rangos admitidos. Una validación detecta los problemas; no decide automáticamente cómo corregirlos.

In [ ]:
pedidos = pd.DataFrame({
    'id': [101, 102, 102],
    'unidades': [2, -1, 3],
    'precio': [10.0, 8.0, None]
})

obligatorias = {'id', 'unidades', 'precio'}
faltantes = obligatorias.difference(pedidos.columns)
if faltantes:
    raise ValueError(f'Faltan columnas: {sorted(faltantes)}')

problemas = pd.DataFrame({
    'id_ausente': pedidos['id'].isna(),
    'id_repetido': pedidos['id'].duplicated(keep=False),
    'unidades_invalidas': pedidos['unidades'].isna() | pedidos['unidades'].le(0),
    'precio_invalido': pedidos['precio'].isna() | pedidos['precio'].lt(0)
})
problemas.loc[problemas.any(axis=1)]



<br>
<div align="right">
<a href="#inicio"><font size=5 color="#B51700"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<div align="right">
<font size=5 color="#B51700"><i class="fa fa-coffee" aria-hidden="true"></i></font>
</div>